In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [2]:
spark = SparkSession.builder.config("spark.sql.streaming.schemaInference", True).getOrCreate()

stream = spark.\
    readStream.\
    format("ws").\
    option("schema", "ticker").\
    load() # we need to pass `option("schema", "ticker")` to get correct channel subscribed

query = stream.select("side", "product_id", "last_size", "best_bid", "best_ask", "time").\
    writeStream.\
    format("console").\
    outputMode("append").\
    option("truncate", "false").\
    start()

query.awaitTermination(10) # Let's wait for 10 seconds.
query.stop() # Let's stop the query
stream.printSchema()
#spark.stop() # And stop the whole session

root
 |-- type: string (nullable = false)
 |-- trade_id: long (nullable = false)
 |-- sequence: long (nullable = false)
 |-- time: timestamp (nullable = false)
 |-- product_id: string (nullable = false)
 |-- price: double (nullable = false)
 |-- side: string (nullable = false)
 |-- last_size: double (nullable = false)
 |-- best_bid: double (nullable = false)
 |-- best_ask: double (nullable = false)



Uruchamiająć `stream.start()` uruchamiamy w osobnym demonie websocket który streamuje wyniki. Jeżeli wystąpi jakiś błąd po stronie front-endu (np. błąd parsowania kolejnej linijki Pythona) fakt ten nie zostanie zgłoszony do sparka i socket pozostanie otwarty! Należy pamiętać, by zamykać stream za każdym razem używająć metody `stop()` (w powyższym przykładzie `query.stop()`). W przypadku utracenia referencji do zapytania, należy zastopować całą sesję również metodą `stop()` (w powyższym przykładzie `spark.stop()`) 

In [3]:
# Panic button - press only if you messed up opening new websocket and lost reference to it

query.stop()
spark.stop()

# Zadanie 1

**Analiza strumienia danych CoinBase (2p)**. Napisz zapytanie, które wypisuje średnią wartość wybranego parametru (np. `price`) w przesuwnych oknach czasowych względem czasu transakcji (kolumna `time`), grupując po relacji wymiany (z jakiej waluty na jaką walutę - kolumna `product_id`). 

# Zadanie 2

**Watermarking i dane opóźnione (2p).** 
Zmodyfikuj zapytanie z zadania 1 tak, aby zademonstować mechanizm znaków wodnych (watermarks) i obsługi danych opóźnionych. W konsoli powinno być widać, że aktualizują się odpowiednie wiersze tabeli wynikowej (tryb update), w szczególności aktualizacja wcześniejszych okien czasowych po przybyciu danych opóźnionych. **Do rozwiązania tego zadania proszę dołączyć przykładowy output i jego opis wyjaśniający na konkretnym przykładzie działanie znaku wodnego i danych opóźnionych**. 

Do ćwiczenia można wykorzystać skrypt w katalogu `/mock` napisany w [Scala-cli](https://scala-cli.virtuslab.org), który posłuży jako kontrolowane źródło danych CoinBase przez Websocket. 

Skrypt można uruchomić wykorzystując Docker:

```
make image
make run
```

Spowoduje to utworzenie websocketowego serwera pod adresem `ws://mock:8025`

Po uruchomieniu serwera należy wykonać poniższą komórkę, w której zapytanie czyta dane z utworzonego websocketa. Skrypt wysyła przykładowe wiadomości w formacie CoinBase co 10 sekund:

- W pierwszej serii wysyłane wiadomości o znacznikach czasowych 0s, 14s, 7s  
- W drugiej serii wysyłane są wiadomości o znacznikach czasowych 15s, 8s, 21s  
- W trzeciej serii wysyłane są wiadomości o znacznikach czasowych 4s, 17s  

Dla tych danych można ustawić okno czasowe na interwał 10 sekund. Skrypt można też zmodyfikować, tak aby wysyłał inne dane. 


In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.config("spark.sql.streaming.schemaInference", True).getOrCreate()

# Note url option!
# we pass explicit url option to subscribe to our mock service
stream = spark.readStream.format("ws").option("schema", "ticker").option("url", "ws://mock:8025").load()
 
# your code here...

# Zadanie 3

**Łączenie strumieni (1p).** Kanał `ticker` zawiera m.in. transakcje z rynków BTC-USD i BTC-EUR. Iloraz cen bitcoina w dolarach i w euro z tej samej chwili to kurs EUR/USD wyznaczony przez rynek kryptowalut.

Połącz każdą transakcję BTC-EUR z transakcjami BTC-USD z poprzedzającej ją sekundy. Użyj złączenia dwóch strumieni (stream-stream join) ze znakami wodnymi na obu strumieniach i warunkiem na czas transakcji (kolumna `time`). Dla każdej pary wypisz czasy i ceny obu transakcji oraz kurs EUR/USD, czyli cenę w USD podzieloną przez cenę w EUR.

Wskazówka: warunek złączenia dwóch strumieni musi zawierać co najmniej jedną równość kolumn.

**Do rozwiązania proszę dołączyć przykładowy output oraz odpowiedzi na pytania:**

- Jak otrzymany kurs ma się do kursu EUR/USD z innego źródła (np. tabeli NBP, kursu referencyjnego EBC lub notowań na żywo)? Skąd biorą się różnice?
- Dlaczego jednej transakcji BTC-EUR odpowiada w wyniku kilka wierszy?
- Mając konta w USD i w EUR, na którym rynku opłaca się teraz kupić BTC? Ile wynosi różnica przy zakupie za 10 000 EUR i czy pokrywa ona koszty transakcji?

**Uwaga o liczbie partycji.** Złączenie strumieni jest operacją stanową. Spark dzieli oba strumienie według klucza złączenia na `spark.sql.shuffle.partitions` partycji, a każda partycja ma własne magazyny stanu (state store), w złączeniu dwóch strumieni po cztery. Domyślnie partycji jest 200, więc zapytanie utrzymuje 800 magazynów stanu i w każdym micro-batchu zapisuje każdy z nich do katalogu checkpoint, nawet pusty. Jeden micro-batch trwa wtedy około 20 sekund, a przy 4 partycjach około 1–2 sekund.

W zapytaniach wsadowych (batch) Spark sam scala małe partycje po etapie shuffle (adaptive query execution). W zapytaniach strumieniowych ten mechanizm jest wyłączony, bo stan jest podzielony na partycje zapisane w checkpoincie i ich liczba nie może się zmieniać między micro-batchami ani po restarcie zapytania. Dlatego komórka poniżej ustawia liczbę partycji na 4:

```python
spark.conf.set("spark.sql.shuffle.partitions", 4)
```

Ustawienie obowiązuje tylko dla zapytań uruchamianych z nowym katalogiem checkpoint. Zapytanie wznowione z istniejącego checkpointu zachowuje pierwotną liczbę partycji. To samo spowolnienie dotyczy agregacji w oknach z zadań 1 i 2, w których każda partycja ma jeden magazyn stanu.


In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.\
    builder.\
    config("spark.sql.streaming.schemaInference", True).\
    config("spark.sql.streaming.forceDeleteTempCheckpointLocation", True).\
    getOrCreate()

# 4 shuffle partitions; with the default 200 each micro-batch takes about 20 s (see the note above)
spark.conf.set("spark.sql.shuffle.partitions", 4)

stream = spark.\
    readStream.\
    format("ws").\
    option("schema", "ticker").\
    load()

# your code...